In [23]:
import json
from collections import defaultdict

# 从文件加载关键词数据
with open('部位描述关键词.json', 'r', encoding='utf-8') as f:
    keyword_description = json.load(f)

with open('诊断结论关键词_hzh.json', 'r', encoding='utf-8') as f:
    keyword_conclusion = json.load(f)

def parse_medical_report(report):
    """解析医学报告，提取各部位的描述和诊断结论"""
    report_lines = report.split('\n')
    parsed_info = {}

    for line in report_lines:
        if line.startswith('- '):
            part = line[2:].split('：')[0]
            try:
                description = line[2:].split('：')[1]
            except:
                continue
            parsed_info[part] = description
        if '诊断结论' in line:
            try:
                diagnosis = line.split('：')[1]
            except:
                continue
            parsed_info['诊断结论'] = diagnosis
            

    return parsed_info

def extract_keywords_from_description(parsed_report, keyword_data):
    """根据关键词数据从报告的部位描述中提取相关信息"""
    extracted_keywords_level1 = []
    extracted_keywords_level2 = []
    extracted_keywords_level3 = []

    for part, description in parsed_report.items():

        description_parts = description.split('，')

        for part_class, part_keywords in keyword_data.items():
            if part not in part_class:
                continue
            
            for keyword_class, keywords in part_keywords.items():
                for keyword, synonym_info in keywords.items():
                    all_synonyms = synonym_info['同义词'] + [keyword]
                    for synonym in all_synonyms:
                        for i, desc in enumerate(description_parts):
                            if '未见' in desc or '无' in desc:
                                continue
                            if '配合不出现' in synonym_info and synonym_info['配合不出现'] in desc:
                                continue
                            if '配合使用' in synonym_info and synonym_info['配合使用'] and synonym_info['配合使用'] not in description:
                                continue
                            if synonym in desc:
                                description_parts[i] = desc.replace(synonym, '')
                                if '含义' in synonym_info and synonym_info['含义']:
                                    extracted_keywords_level1.append(f"{part}-{synonym_info['含义']}")
                                else:
                                    extracted_keywords_level1.append(f"{part}-{keyword}")

                                if keyword_class != '病变类':
                                    extracted_keywords_level2.append(f"{part}-{keyword_class}")
                                else:
                                    if '含义' in synonym_info and synonym_info['含义']:
                                        extracted_keywords_level2.append(f"{part}-{synonym_info['含义']}")
                                    else:
                                        extracted_keywords_level2.append(f"{part}-{keyword}")

                                extracted_keywords_level3.append(f"{part}-异常")

    return extracted_keywords_level1, extracted_keywords_level2, extracted_keywords_level3

def extract_keywords_from_conclusion(parsed_report, keyword_data):
    """根据关键词数据从报告的诊断结论中提取相关信息"""
    extracted_keywords_level1 = []
    extracted_keywords_level2 = []
    extracted_keywords_level3 = []

    if '诊断结论' not in parsed_report:
        return extracted_keywords_level1, extracted_keywords_level2, extracted_keywords_level3

    conclusion = parsed_report['诊断结论']
    conclusion_parts = conclusion.split('；')

    for part_or_disease, part_keywords in keyword_data.items():
        if part_or_disease == "子部位":
            continue
        for keyword_class, keywords in part_keywords.items():
            for keyword, synonym_info in keywords.items():
                all_synonyms = synonym_info['同义词'] + [keyword]
                for synonym in all_synonyms:
                    for i, conclusion in enumerate(conclusion_parts):
                        if '未见' in conclusion or '无' in conclusion:
                            continue
                        if '配合不出现' in synonym_info and synonym_info['配合不出现'] in conclusion:
                            continue
                        if '配合使用' in synonym_info and synonym_info['配合使用'] and synonym_info['配合使用'] not in conclusion:
                            continue
                        if synonym in conclusion:
                            conclusion_parts[i] = conclusion.replace(synonym, '')
                            if '含义' in synonym_info and synonym_info['含义']:
                                extracted_keywords_level1.append(f"诊断结论-{synonym_info['含义']}")
                            else:
                                extracted_keywords_level1.append(f"诊断结论-{keyword}")

                            if keyword_class != '病变类':
                                extracted_keywords_level2.append(f"诊断结论-{keyword_class}")
                            else:
                                conclusion_parts[i] = conclusion.replace(synonym, '')
                                if '含义' in synonym_info and synonym_info['含义']:
                                    extracted_keywords_level2.append(f"诊断结论-{synonym_info['含义']}")
                                else:
                                    extracted_keywords_level2.append(f"诊断结论-{keyword}")
                            
                            extracted_keywords_level3.append(f"诊断结论-异常")

    return extracted_keywords_level1, extracted_keywords_level2, extracted_keywords_level3

def calculate_metrics(pred_keywords, label_keywords):
    """计算精确度、召回率和F1分数"""
    set_pred = set(pred_keywords)
    set_label = set(label_keywords)

    if not set_pred and not set_label:
        return 1.0, 1.0, 1.0  # 如果两者都为空，定义为完美匹配

    tp = len(set_pred.intersection(set_label))
    fp = len(set_pred - set_label)
    fn = len(set_label - set_pred)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

import json

# 假设 `parse_medical_report` 和 `extract_keywords_from_description` 已经定义

# 加载数据

with open('/work/model/xieqiang/推理和评测/union_2018_2019_2020_2021_train_sampled_review_42506cases_9epoch/results_ascend_llama_factory_huaxi_union_2022_05_to_2023_test_14694cases-1660.json', 'r', encoding='utf-8') as f:
    test_results = json.load(f)

# 用于存储计算出的精确度、召回率和F1分数
all_precision_dic = defaultdict(list)
all_recall_dic = defaultdict(list)
all_f1_dic = defaultdict(list)

print(len(test_results))
# 逐个处理每个case
for case in test_results:
    pred = case['pred']
    # print(pred)
    label = case['label']

    # 提取pred和label的关键词
    parse_pred = parse_medical_report(pred)
    extracted_desc_pred_level1, extracted_desc_pred_level2, extracted_desc_pred_level3 = extract_keywords_from_description(parse_pred, keyword_description)
    extracted_conclu_pred_level1, extracted_conclu_pred_level2, extracted_conclu_pred_level3 = extract_keywords_from_conclusion(parse_pred, keyword_conclusion)
    # print(extracted_info_pred)
    parse_label = parse_medical_report(label)
    extracted_desc_label_level1, extracted_desc_label_level2, extracted_desc_label_level3 = extract_keywords_from_description(parse_label, keyword_description)
    extracted_conclu_label_level1, extracted_conclu_label_level2, extracted_conclu_label_level3 = extract_keywords_from_conclusion(parse_label, keyword_conclusion)
    # 转换为集合，方便计算交集
    # print(extracted_info_label)
    # if case['image_id'] == "01.0000001262430":
    #     print("pred",pred)
    #     print("label",label)
    #     print("extracted_info_pred",extracted_info_pred)
    #     print("extracted_info_label",extracted_info_label)

    # 计算 Precision, Recall 和 F1 分数
    
    # 存储结果
    overall_level1_precision, overall_level1_recall, overall_level1_f1 = calculate_metrics(
        extracted_desc_pred_level1 + extracted_conclu_pred_level1,
        extracted_desc_label_level1 + extracted_conclu_label_level1
    )

    desc_level1_precision, desc_level1_recall, desc_level1_f1 = calculate_metrics(
        extracted_desc_pred_level1,
        extracted_desc_label_level1
    )

    conclu_level1_precision, conclu_level1_recall, conclu_level1_f1 = calculate_metrics(
        extracted_conclu_pred_level1,
        extracted_conclu_label_level1
    )

    overall_level2_precision, overall_level2_recall, overall_level2_f1 = calculate_metrics(
        extracted_desc_pred_level2 + extracted_conclu_pred_level2,
        extracted_desc_label_level2 + extracted_conclu_label_level2
    )

    overall_level3_precision, overall_level3_recall, overall_level3_f1 = calculate_metrics(
        extracted_desc_pred_level3 + extracted_conclu_pred_level3,
        extracted_desc_label_level3 + extracted_conclu_label_level3
    )
    all_precision_dic['overall_level1'].append(overall_level1_precision)
    all_recall_dic['overall_level1'].append(overall_level1_recall)
    all_f1_dic['overall_level1'].append(overall_level1_f1)
    all_precision_dic['desc_level1'].append(desc_level1_precision)
    all_recall_dic['desc_level1'].append(desc_level1_recall)
    all_f1_dic['desc_level1'].append(desc_level1_f1)
    all_precision_dic['conclu_level1'].append(conclu_level1_precision)
    all_recall_dic['conclu_level1'].append(conclu_level1_recall)
    all_f1_dic['conclu_level1'].append(conclu_level1_f1)
    all_precision_dic['overall_level2'].append(overall_level2_precision)
    all_recall_dic['overall_level2'].append(overall_level2_recall)
    all_f1_dic['overall_level2'].append(overall_level2_f1)
    all_precision_dic['overall_level3'].append(overall_level3_precision)
    all_recall_dic['overall_level3'].append(overall_level3_recall)
    all_f1_dic['overall_level3'].append(overall_level3_f1)
    
    # 打印输出
    # order = ['食道', '贲门', '胃底', '胃体', '胃角', '胃窦', '幽门', '十二指肠球部', '十二指肠降部', '诊断结论']
    # print(f"ID: {case['image_id']}")
    # print(f"Pred: {pred} \n")
    # print(f"Label: {label} \n")
    # print(f"Pred Extracted Info: {sorted(list(set_pred), key=lambda x: order.index(x.split('-')[0]))}")
    # print(f"Label Extracted Info: {sorted(list(set_label), key=lambda x: order.index(x.split('-')[0]))}")
    # print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")
    # print("========================================")
    # break

# 计算总体的平均精确度、召回率和F1分数

for key in all_precision_dic.keys():    
    avg_precision = sum(all_precision_dic[key]) / len(all_precision_dic[key])
    avg_recall = sum(all_recall_dic[key]) / len(all_recall_dic[key])
    avg_f1 = sum(all_f1_dic[key]) / len(all_f1_dic[key])
    print(f"{key} - Average Precision: {avg_precision:.4f}, Average Recall: {avg_recall:.4f}, Average F1 Score: {avg_f1:.4f}")

import pandas as pd

# 创建一个空的列表用于存储数据
data = []

# 遍历所有的key并计算相应的平均值
for key in all_precision_dic.keys():
    avg_precision = sum(all_precision_dic[key]) / len(all_precision_dic[key])
    avg_recall = sum(all_recall_dic[key]) / len(all_recall_dic[key])
    avg_f1 = sum(all_f1_dic[key]) / len(all_f1_dic[key])
    data.extend([avg_precision, avg_recall, avg_f1])

# 使用pandas创建DataFrame
df = pd.DataFrame([data])

df


14694
overall_level1 - Average Precision: 0.6559, Average Recall: 0.5648, Average F1 Score: 0.5876
desc_level1 - Average Precision: 0.6213, Average Recall: 0.5135, Average F1 Score: 0.5402
conclu_level1 - Average Precision: 0.7374, Average Recall: 0.7079, Average F1 Score: 0.6944
overall_level2 - Average Precision: 0.7393, Average Recall: 0.6397, Average F1 Score: 0.6639
overall_level3 - Average Precision: 0.9492, Average Recall: 0.8616, Average F1 Score: 0.8927


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0.65591,0.564787,0.587587,0.621272,0.513454,0.540178,0.737411,0.707905,0.694393,0.739259,0.639679,0.663856,0.949154,0.861629,0.892682


[0.0,
 0.0,
 0.0,
 0.0,
 0.125,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.6,
 0.0,
 0.3333333333333333,
 0.0,
 0.0,
 0.0,
 0.6666666666666666,
 0.0,
 0.0,
 0.0,
 0.0,
 0.42857142857142855,
 0.0,
 0.0,
 0.0,
 0.6666666666666666,
 0.25,
 0.0,
 0.0,
 0.0,
 0.0,
 0.42857142857142855,
 0.3333333333333333,
 0.6,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.42857142857142855,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.375,
 0.2,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.5,
 0.2,
 0.25,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.42857142857142855,
 0.5,
 0.5,
 0.42857142857142855,
 0.3333333333333333,
 0.0,
 0.0,
 0.42857142857142855,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.6666666666666666,
 0.5714285714285714,
 0.0,
 0.6,
 0.42857142857142855,
 0.0,
 0.0,
 0.0,
 0.14285714285714285,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.16666666666666666,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 

In [5]:
from pycocoevalcap.eval import COCOEvalCap
from pycocotools.coco import COCO
import json
import re
import os
# from bert_score import score

def coco_caption_eval(results_file, annotation_file):
    img_ids = []
    with open(annotation_file, 'r', encoding='utf-8') as f:
        ann_data = json.load(f)
        for ann in ann_data['annotations']:
            img_ids.append(ann['image_id'])

    img_ids = set(img_ids)
    coco = COCO(annotation_file)
    coco_result = coco.loadRes(results_file)

    # create coco_eval object by taking coco and coco_result
    coco_eval = COCOEvalCap(coco, coco_result)

    # evaluate on a subset of images by setting
    if img_ids:
        coco_eval.params['image_id'] = coco_result.getImgIds()

    coco_eval.evaluate()

    return coco_eval

# def calculate_bertscore(testepochfile):
#     with open(testepochfile, 'r', encoding='utf-8') as f:
#         data = json.load(f)

#     hypothesis, references = [], []
#     for item in data:
#         hypothesis.append(item['pred'])
#         references.append(item['label'])

#     P, R, F1 = score(hypothesis, references, lang='zh', verbose=True, model_type='bert-base-chinese')
    
#     return F1

# path = 'union_2018_2019_2020_2021_train_sampled_review_42506cases_9epoch'
# for file in os.listdir(path):
#     if not file.endswith('json') or not 'llama' in file:
#         continue
#     if '3100' in file:
#         annotation_file = 'ascend_llama_factory_huaxi_union_2022_test_sampled_3100cases_gt.json'
#     elif '14694' in file:
#         annotation_file = 'ascend_llama_factory_huaxi_union_2022_05_to_2023_test_14694cases_gt.json'
#     elif '5000' in file:
#         annotation_file = 'ascend_llama_factory_huaxi_union_2018_2019_2020_2021_test_review_5000cases_gt.json'
#     elif '7091' in file:
#         annotation_file = 'ascend_llama_factory_hfyy_union_2023_2024_test_7091cases_gt.json'
#     elif '1122' in file:
#         annotation_file = 'ascend_llama_factory_hfcas_union_2022_05_to_2023_test_1122cases_gt.json'
#     eval_result_file = os.path.join(path, file)

#     print(f"Evaluating {eval_result_file} against {annotation_file}")
#     coco_eval = coco_caption_eval(eval_result_file, annotation_file)
#     # bert_f1 = calculate_bertscore(eval_result_file)

#     for metric, score in coco_eval.eval.items():
#         print(f"{metric}: {score:.4f}")
#     # print(f"BERTScore F1: {bert_f1.mean():.4f}")

#     print("========================================")

coco_eval = coco_caption_eval('hulu_32B_hfyy_union_2023_2024_test_7091cases.json', 'ascend_llama_factory_hfyy_union_2023_2024_test_7091cases_gt.json')
for metric, score in coco_eval.eval.items():
    print(f"{metric}: {score:.4f}")

coco_eval.output_scores

loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.08s)
creating index...
index created!
tokenization...
setting up scorers...
computing Bleu score...
{'testlen': 532887, 'reflen': 592343, 'guess': [532887, 525796, 518705, 511614], 'correct': [200072, 65190, 18844, 9498]}
ratio: 0.8996257236094611
computing METEOR score...
computing Rouge score...
computing CIDEr score...
Bleu: 0.3358
METEOR: 0.2238
ROUGE_L: 0.5168
CIDEr: 0.0097


{'Bleu-1': [0.33293029856692663,
  0.23198762209736667,
  0.26027397259917434,
  0.26991968298250657,
  0.32031138532223874,
  0.29610240945779653,
  0.34117647058422146,
  0.29416082748578476,
  0.3322382608291378,
  0.31801222219447145,
  0.34154603703491443,
  0.3148148148118999,
  0.26388888888522377,
  0.36001200755267715,
  0.2821511409245846,
  0.3239816440037519,
  0.1923076923058432,
  0.28409090908768075,
  0.3963249409312148,
  0.34923540286614846,
  0.1976744186023526,
  0.21680843879956505,
  0.2666666666637037,
  0.31801222219447145,
  0.35294117646643597,
  0.32385056615976965,
  0.23112636445311663,
  0.3249999999959375,
  0.3188745867231765,
  0.3333333333287037,
  0.3378378378332724,
  0.3709615866084704,
  0.3589443470372261,
  0.3026853454335988,
  0.2763157894700485,
  0.2329815024052435,
  0.3650793650735702,
  0.2585721425164341,
  0.3207984582743068,
  0.3246753246711081,
  0.31428571427673463,
  0.3208256795267023,
  0.2413793103420531,
  0.3888888888845679,
  

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, pearsonr
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

# 设置本地字体
font_path = 'msyh.ttf'  # 请替换为您字体文件的实际路径
prop = font_manager.FontProperties(fname=font_path)

data = dict(coco_eval.output_scores | all_precision_dic)
del data['desc_level1'] 
del data['conclu_level1']
df = pd.DataFrame(data)

# 计算皮尔逊相关系数矩阵
pearson_corr_matrix = df.corr(method='pearson')

# 进行双侧Wilcoxon符号秩检验（两两比较）
wilcoxon_results = {}
for col1 in df.columns:
    for col2 in df.columns:
        if col1 != col2:
            # 计算Wilcoxon符号秩检验
            stat, p_value = wilcoxon(df[col1], df[col2])
            wilcoxon_results[(col1, col2)] = p_value

# 添加星号标记
def add_stars(p_value):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return ''

# 创建一个包含星号的矩阵
star_matrix = pd.DataFrame(np.zeros_like(pearson_corr_matrix, dtype=object), columns=pearson_corr_matrix.columns, index=pearson_corr_matrix.columns)
for (col1, col2), p_value in wilcoxon_results.items():
    star_matrix.loc[col1, col2] = add_stars(p_value)

# 创建下三角掩码，确保对角线部分不被遮挡
mask = np.triu(np.ones_like(pearson_corr_matrix, dtype=bool), k=1)

colors = ["#DBE5EA", "#F99884", "#B02625"]
cmap = LinearSegmentedColormap.from_list("custom_orange_blue", colors, N=256)

# 设置绘制热力图的大小
plt.figure(figsize=(8, 6), dpi=100)

# 绘制热力图，使用自定义的渐变颜色
sns.heatmap(pearson_corr_matrix, mask=mask, annot=True, fmt='.2f', cmap=cmap, 
            cbar_kws={'label': 'Pearson correlation'}, annot_kws={"size": 8, "ha": "center", "color": "black"},  # 设置数字左对齐
            xticklabels=pearson_corr_matrix.columns, yticklabels=pearson_corr_matrix.columns, 
            linewidths=0.5, linecolor='black', cbar=True)

# 获取当前坐标轴
ax = plt.gca()

# 创建一个三角形，覆盖右上三角的区域
triangle = patches.Polygon(((1, 0.086), (1, 1), (0.086, 1)), closed=True, color='white', transform=ax.transAxes, zorder=5)
ax.add_patch(triangle)

# 在热力图上添加星号
for i in range(len(pearson_corr_matrix.columns)):
    for j in range(i):
        plt.text(j + 0.8, i + 0.3, star_matrix.iloc[i, j], ha='center', va='center', color='black', fontsize=6)

# 设置标题
plt.title('皮尔逊相关系数下三角热力矩阵（Wilcoxon p值）', fontproperties=prop)
plt.xticks(rotation=45, ha='right')

# 显示图像
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, wilcoxon
from matplotlib import font_manager

# 设置本地字体
font_path = 'msyh.ttf'  # 请替换为您字体文件的实际路径
prop = font_manager.FontProperties(fname=font_path)

# 转换为 DataFrame 便于操作
df = pd.DataFrame(coco_eval.output_scores | all_precision_dic)

# 计算皮尔逊相关系数
pearson_corr_matrix = df.corr(method='pearson')

# 打印皮尔逊相关系数矩阵
print("皮尔逊相关系数矩阵：")
print(pearson_corr_matrix)

# 进行双侧Wilcoxon符号秩检验（两两比较）
wilcoxon_results = {}
for col1 in df.columns:
    for col2 in df.columns:
        if col1 != col2:
            # 计算Wilcoxon符号秩检验
            stat, p_value = wilcoxon(df[col1], df[col2])
            wilcoxon_results[(col1, col2)] = p_value

# 打印Wilcoxon检验的p值
print("\nWilcoxon符号秩检验结果（p值）：")
for key, p_value in wilcoxon_results.items():
    print(f"{key}: p-value = {p_value}")

# 绘制散点图矩阵
# sns.pairplot(df)

# 手动绘制拟合直线和标注 Pearson r 和 p-value
for i in range(len(df.columns)):
    for j in range(i+1, len(df.columns)):
        col1 = df.columns[i]
        col2 = df.columns[j]
        
        # 计算皮尔逊相关系数和p-value
        r, p_value = pearsonr(df[col1], df[col2])
        if p_value < 1e-18:
            p_value_str = r'$< 10^{-18}$'  # 角标格式，不加粗
        else:
            mantissa, exponent = f'{p_value:.2e}'.split('e')
            p_value_str = r'$' + mantissa + r' \times 10^{'+exponent+'}$'     
           
        # 创建新的图形并绘制拟合直线
        plt.figure(figsize=(6, 4), dpi=100)
        sns.regplot(x=df[col1], y=df[col2], 
                    line_kws={'color': '#76A8A5'},  # 拟合线的颜色
                    scatter_kws={'color': '#B7D8B3', 'alpha': 0.6, 'edgecolor': '#A0C1A4'})

        # 添加文本框显示 Pearson r 和 p-value
        ax = plt.gca()
        ax.annotate(f"Pearson's r: {r:.2f}\np-value: {p_value_str}",
                    xy=(0.18, 0.87), xycoords='axes fraction',
                    ha='center', va='center', fontsize=12,
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=1))
        plt.title(f'{col1} vs {col2}', fontproperties=prop)
        plt.grid(True)
        plt.show()
#B7D3D3
# 展示散点图矩阵标题
plt.suptitle('散点图矩阵', y=1.02, fontproperties=prop)
plt.show()

In [ ]:
# import os
# os.environ["OPENBLAS_NUM_THREADS"] = "4"  # 你可以根据需要设置线程数

# from bert_score import score
# import json
# eval_result_file = 'union_2018_2019_2020_2021_train_sampled_review_42506cases_9epoch/results_ascend_llama_factory_huaxi_union_2022_test_sampled_3100cases-2997.json'


# def calculate_bertscore(testepochfile):
#     with open(testepochfile, 'r', encoding='utf-8') as f:
#         data = json.load(f)

#     hypothesis, references = [], []
#     for item in data:
#         hypothesis.append(item['pred'])
#         references.append(item['label'])

#     P, R, F1 = score(hypothesis, references, lang='zh', verbose=True, model_type='bert-base-chinese')
    
#     return F1

# calculate_bertscore(eval_result_file)

In [ ]:
# import json

# test_file = 'ascend_llama_factory_huaxi_union_2022_test_sampled_3100cases.json'
# gt_file = 'ascend_llama_factory_huaxi_union_2022_test_sampled_3100cases_gt.json'

# with open(test_file, 'r', encoding='utf-8') as f:
#     data = json.load(f)

# gt_data = {'annotations': [], 'images': []}

# for item in data:
#     image_id = item['id']
#     caption = item['messages'][1]['content']
#     id = image_id

#     gt_data['annotations'].append({
#         'image_id': image_id,
#         'caption': caption,
#         'id': id
#     })
#     gt_data['images'].append({
#         'id': image_id,
#     })

# with open(gt_file, 'w', encoding='utf-8') as f:
#     json.dump(gt_data, f, ensure_ascii=False, indent=4)
